# Imports

In [1]:
import warnings
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
FOLDS = 5
ID = 'id'
TARGET = 'class'
TARGET_MAPPING = {
    "GALAXY": 0,
    "QSO": 1,
    "STAR": 2
}
TARGET_INV_MAPPING = {
    0:"GALAXY",
    1:"QSO",
    2:"STAR"
}
RUN_OPTUNA = False
RUN_SKF = True

# Data Loading & Original Dataset Preparation

In [ ]:
train = pd.read_csv("playground-series-s6e6\\train.csv")
test = pd.read_csv("playground-series-s6e6\\test.csv")
original = pd.read_csv("playground-series-s6e6\\externel_dataset\\star_classification.csv") 

# train_id = train[ID]
# test_id = test[ID]

# def get_spectral_type(g, r):
#     return pd.cut(
#         r - g, 
#         [-np.inf, -1, -0.5, 0, np.inf],
#         labels=['M', 'G/K', 'A/F', 'O/B']
#     ).astype(str)

# def get_galaxy_population(u, r):
#     return pd.cut(
#         u - r, 
#         [-np.inf, 1.4, 2.2, np.inf],
#         labels=['Blue_Cloud', 'Green_Valley', 'Red_Sequence']
#     ).astype(str)

# original['spectral_type'] = get_spectral_type(original['g'], original['r'])
# original['galaxy_population'] = get_galaxy_population(original['u'], original['r'])

# target_map = {'GALAXY': 'GALAXY', 'QSO': 'QSO', 'STAR': 'STAR'}
# original[TARGET] = original[TARGET].map(target_map)
# train[TARGET] = train[TARGET].map(target_map)

# base_features = [col for col in train.columns if col not in [TARGET, ID]]
# cat_cols_init = train.drop(columns=[ID, TARGET]).select_dtypes(include=['object']).columns.tolist()
# num_cols_init = [c for c in train.drop(columns=[ID, TARGET]).select_dtypes(exclude=['object']).columns]

# original_numeric_target = original[TARGET].map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})
# original_global_mean = original_numeric_target.mean()
# original_global_median = original_numeric_target.median()

# original_stats = {}
# for col in base_features:
#     if col in original.columns:
#         orig_temp = original[[col]].copy()
#         orig_temp[TARGET] = original_numeric_target
#         if col in num_cols_init:
#             orig_temp[col] = np.floor(orig_temp[col])
            
#         stats = orig_temp.groupby(col)[TARGET].agg(['mean', 'median', 'std', 'skew', 'count']).reset_index()
#         stats.columns = [col] + [f"orig_{col}_{s}" for s in ['mean', 'median', 'std', 'skew', 'count']]
#         original_stats[col] = stats

# Feature Engineering Pipeline Definition

In [4]:
ID = 'id'
TARGET = 'class'
train[TARGET] = train[TARGET].map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
color_pairs = [
    ('u', 'g'),
    ('g', 'r'),
    ('i', 'z'),
    ('r', 'z'),
    ('u', 'z'),
]
important_combos = [
    ('alpha_cat_', 'delta_cat_'),
    ('u_cat_', 'z_cat_'),
]
important_combos = sorted(important_combos)
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_g_/_redshift'] = (df['g'] / (df['redshift'] + 1e-6)).astype('float32')
    df['_i_/_redshift'] = (df['i'] / (df['redshift'] + 1e-6)).astype('float32')
    df['_Distance_Modulus'] = (6.0 * np.log10(np.abs(df['redshift']) + 1e-6)).astype('float32')
    for a, b in color_pairs:
        df[f"_{a}-{b}"] = (df[a] - df[b]).astype('float32')

    # Categorize string cats
    for col in cat_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')

    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')

    # Discretize numericals
    bin_config = {'delta': [100, 500]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype('category')

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, new_cat_cols, new_num_cols, combo_names = feature_engineering(X_test, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

cat_cols = sorted(cat_cols)
X = X.reindex(sorted(X.columns), axis=1)
X_test = X_test.reindex(sorted(X_test.columns), axis=1)
print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

X      init shape: (577347, 10)
X_test init shape: (247435, 10) 

init len(cat_cols): 2
init len(num_cols): 8 

len(new_cat_cols): 12
len(new_num_cols): 8 

prep len(cat_cols): 14
prep len(num_cols): 16 

X      prep shape: (577347, 30)
X_test prep shape: (247435, 30) 



# Feature Engineering Execution 

In [5]:
# X_train = train.drop(columns=[ID, TARGET])
# y_train = train[TARGET]
# X_test = test.drop(columns=[ID])

# chosen_strategies = ['encoding', 'colors', 'ratios', 'redshift', 'position', 'interactions', 'flux']

# SKEW_THRESHOLD = 1.5
# UNIQUENESS_THRESHOLD = 15
# current_num_cols = X_train.select_dtypes(exclude=['category', 'object']).columns.tolist()

# for col in current_num_cols:
#     if X_train[col].nunique() <= UNIQUENESS_THRESHOLD: continue
#     if abs(X_train[col].skew()) > SKEW_THRESHOLD:
#         X_train[col] = (np.sign(X_train[col]) * np.log1p(np.abs(X_train[col]))).astype('float32')
#         X_test[col] = (np.sign(X_test[col]) * np.log1p(np.abs(X_test[col]))).astype('float32')

# for col in X_train.columns.tolist():
#     if isinstance(X_train[col].dtype, pd.CategoricalDtype) or X_train[col].dtype == 'object' or X_train[col].nunique() <= UNIQUENESS_THRESHOLD:
#         X_train[col], X_test[col] = pd.Categorical(X_train[col]), pd.Categorical(X_test[col])

# constant_cols = [col for col in X_train.columns if X_train[col].nunique() <= 1]
# if constant_cols:
#     X_train.drop(columns=constant_cols, inplace=True)
#     X_test.drop(columns=constant_cols, inplace=True)

## Data Processing

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

def ordinal_encode(train_df, test_df=None):
    train_df = train_df.copy()

    cat_cols = train_df.select_dtypes(include=['object', bool, 'category']).columns.tolist()

    encoder = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )

    if test_df is not None:
        test_df = test_df.copy()

        encoder.fit(
            pd.concat(
                [train_df[cat_cols], test_df[cat_cols]],
                axis=0
            )
        )

        train_df[cat_cols] = encoder.transform(train_df[cat_cols])
        test_df[cat_cols] = encoder.transform(test_df[cat_cols])

        return train_df, test_df, encoder

    encoder.fit(train_df[cat_cols])
    train_df[cat_cols] = encoder.transform(train_df[cat_cols])

    return train_df, encoder


train_df_enc, test_df_enc, encoder = ordinal_encode(X, X_test)

## Optuna Search

In [22]:
import lightgbm as lgb
import optuna
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

print(lgb.__version__)

X_train_, X_test_, y_train_, y_test_ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lgb_best_params = {
    'n_estimators': 1841,
    'learning_rate': 0.016121467631636684,
    'num_leaves': 126,
    'max_depth': 12,
    'min_child_samples': 7,
    'subsample': 0.9137011411523339,
    'colsample_bytree': 0.6472035788104408,
    'reg_alpha': 1.6143965568012781,
    'reg_lambda': 0.0457226857453295,
    'min_split_gain': 0.520062046419683,
    "max_bin": 63
}

def objective(trial):

    params = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "device": "gpu",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            max(1200, int(lgb_best_params["n_estimators"] * 0.8)),
            min(2500, int(lgb_best_params["n_estimators"] * 1.2))
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            lgb_best_params["learning_rate"] * 0.7,
            lgb_best_params["learning_rate"] * 1.3,
            log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            max(31, lgb_best_params["num_leaves"] - 20),
            min(255, lgb_best_params["num_leaves"] + 20)
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            max(6, lgb_best_params["max_depth"] - 2),
            lgb_best_params["max_depth"] + 2
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            max(2, lgb_best_params["min_child_samples"] - 3),
            lgb_best_params["min_child_samples"] + 3
        ),

        "subsample": trial.suggest_float(
            "subsample",
            max(0.7, lgb_best_params["subsample"] - 0.10),
            min(1.0, lgb_best_params["subsample"] + 0.05)
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            max(0.4, lgb_best_params["colsample_bytree"] - 0.10),
            min(1.0, lgb_best_params["colsample_bytree"] + 0.10)
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            max(1e-4, lgb_best_params["reg_alpha"] * 0.3),
            lgb_best_params["reg_alpha"] * 3,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            max(1e-5, lgb_best_params["reg_lambda"] * 0.2),
            lgb_best_params["reg_lambda"] * 5,
            log=True
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            max(0.0, lgb_best_params["min_split_gain"] - 0.30),
            lgb_best_params["min_split_gain"] + 0.30
        ),

        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
        "max_bin": 63
    }

    model = lgb.LGBMClassifier(**params)

    model.fit(X_train_, y_train_)
    y_pred = model.predict(X_test_)
    b_acc = balanced_accuracy_score(y_test_, y_pred)
    print(f"Trial {trial.number} | BAL_ACC: {b_acc:.6f}")

    return b_acc


def best_trial_callback(study, trial):

    print("\n" + "=" * 60)
    print(f"Trial {trial.number} Completed")
    print(f"Trial BAL_ACC: {trial.value:.6f}")
    print(f"\nBest BAL_ACC So Far: {study.best_value:.6f}")
    print("\nBest Parameters So Far:")
    print(study.best_params)
    print("=" * 60)


if RUN_OPTUNA:
    sampler = optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=5,
        multivariate=True,
        group=True
    )

    study_lgb = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="lightgbm_local_search"
    )

    study_lgb.enqueue_trial(lgb_best_params)

    study_lgb.optimize(
        objective,
        n_trials=100,
        callbacks=[best_trial_callback],
        show_progress_bar=True
    )

    print("\nBest BAL_ACC:")
    print(study_lgb.best_value)

    print("\nBest Parameters:")
    print(study_lgb.best_params)

4.6.0


## Best LightGBM 

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import lightgbm as lgb

SEED = 42
RUN_SKF = True
N_FOLDS = 5

best_lgb_params = {'n_estimators': 1402, 'learning_rate': 0.015683531088405856, 'num_leaves': 138, 'max_depth': 15, 'min_child_samples': 9, 'subsample': 0.900228907109458, 'colsample_bytree': 0.6185476387342294, 'reg_alpha': 7.804739536508066, 'reg_lambda': 0.06876460724759273, 'min_split_gain': 0.08356013475510977}

best_lgb_params['class_weight']='balanced'
best_lgb_params['random_state']=42
best_lgb_params['max_bin']=63
best_lgb_params['objective']="multiclass"
best_lgb_params["metric"] = "multi_logloss"

if RUN_SKF:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_scores = []

    print(f"{'Fold':<6} {'Balanced Accuracy':^20}")
    print("-" * 30)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_, y_train_), 1):
        X_tr, X_val = X_train_.iloc[train_idx], X_train_.iloc[val_idx]
        y_tr, y_val = y_train_.iloc[train_idx], y_train_.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
        **best_lgb_params,
        verbose=-1
        )
        
        model.fit(X_tr, y_tr)
        
        y_pred = model.predict(X_val)
        balanced_acc = balanced_accuracy_score(y_val, y_pred)
        fold_scores.append(balanced_acc)
        
        print(f"{fold:<6} {balanced_acc:>20.4f}")

    print("-" * 30)
    print(f"{'Mean':<6} {np.mean(fold_scores):>20.4f}")
    print(f"{'Std':<6} {np.std(fold_scores):>20.4f}")

Fold    Balanced Accuracy  
------------------------------


## Submission

In [ ]:
lgb_model = lgb.LGBMClassifier(
    **best_lgb_params, 
    verbose=-1
    )
lgb_model.fit(X, y)

FILE_NAME = "lgb_21_06_2026_new_features_v2"
SUB_FILE_NAME = f"submissions\\{FILE_NAME}.csv"
PROB_FILE_NAME = f"probabilities\\{FILE_NAME}.csv"

probabilities = lgb_model.predict_proba(X_test)
class_names = ['GALAXY', 'QSO', 'STAR'] 
proba_df_classes = pd.DataFrame(probabilities, columns=class_names)
proba_df = pd.concat([test[ID].reset_index(drop=True), proba_df_classes], axis=1)
proba_df.to_csv(PROB_FILE_NAME, index=False)

predictions = pd.Series(lgb_model.predict(X_test)).map(TARGET_INV_MAPPING)
sub_df = pd.DataFrame({ID:test[ID], TARGET:predictions})
sub_df.to_csv(SUB_FILE_NAME, index=False)
sub_df.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [ ]:
# LightGBM BENCHMARK
# Fold    Balanced Accuracy  
# ------------------------------
# 1                    0.9632
# 2                    0.9640
# 3                    0.9628
# 4                    0.9628
# 5                    0.9631
# ------------------------------
# Mean                 0.9632
# Std                  0.0005